In [1]:
from pathlib import Path

import pandas as pd

from src.thesis_analysis_report import (
    build_comparison_table,
    export_outputs,
    load_results,
    pareto_frontier,
    pick_best_by_method,
)

In [12]:
REPO_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd() / "src").exists() else Path(Path.cwd()).resolve()
REPO_ROOT = REPO_ROOT.parent
INPUT_DIRS = [
    REPO_ROOT /"src"/ "experiments",
    REPO_ROOT / "src" / "indexing" / "data" / "benchmarks",
]
OUTPUT_DIR = REPO_ROOT / "experiments" / "results"
PRIMARY_METRIC = "Precision@10"

In [13]:
INPUT_DIRS

[WindowsPath('D:/P_work/Rag-VKR/src/experiments'),
 WindowsPath('D:/P_work/Rag-VKR/src/indexing/data/benchmarks')]

In [14]:
# Load results

df_all = load_results(INPUT_DIRS)
df_all.head()

,method,m,ef_construct,ef_search,timestamp,Precision@1,Precision@3,Precision@5,Precision@10,Recall@10,...,n_failed,k1,b,error,rrf_k,hnsw_collection,_source_file,avg_latency_s,avg_latency_ms,success_rate
0,hnsw,8.0,100.0,50.0,2026-03-29 10:43:53,0.014,0.662667,0.7896,0.8824,0.8824,...,0,NaN,NaN,NaN,NaN,NaN,D:\P_work\Rag-VKR\src\indexing\data\benchmarks...,0.011506,11.5064,1.0
1,hnsw,8.0,100.0,100.0,2026-03-29 10:44:04,0.024,0.663333,0.7920,0.8838,0.8838,...,0,NaN,NaN,NaN,NaN,NaN,D:\P_work\Rag-VKR\src\indexing\data\benchmarks...,0.011186,11.1858,1.0
2,hnsw,8.0,100.0,200.0,2026-03-29 10:44:15,0.010,0.657333,0.7872,0.8848,0.8848,...,0,NaN,NaN,NaN,NaN,NaN,D:\P_work\Rag-VKR\src\indexing\data\benchmarks...,0.011239,11.2394,1.0
3,hnsw,8.0,200.0,50.0,2026-03-29 10:44:26,0.022,0.662667,0.7908,0.8842,0.8842,...,0,NaN,NaN,NaN,NaN,NaN,D:\P_work\Rag-VKR\src\indexing\data\benchmarks...,0.011867,11.8666,1.0
4,hnsw,8.0,200.0,100.0,2026-03-29 10:44:38,0.018,0.661333,0.7896,0.8840,0.8840,...,0,NaN,NaN,NaN,NaN,NaN,D:\P_work\Rag-VKR\src\indexing\data\benchmarks...,0.012111,12.1106,1.0


In [16]:
# Basic overview

(df_all.groupby("method").size().rename("n_runs").reset_index().sort_values("n_runs", ascending=False))

,method,n_runs
1,hnsw,36
0,bm25,12
2,hybrid_rrf,4


In [17]:
# Filter successful runs only (if available)

df_ok = df_all.copy()
if "n_failed" in df_ok.columns:
    df_ok = df_ok[df_ok["n_failed"].fillna(0) == 0]

(df_ok.groupby("method").size().rename("n_success_runs").reset_index())

,method,n_success_runs
0,hnsw,36
1,hybrid_rrf,4


In [18]:
# Best parameters for each method (by primary metric)

best = pick_best_by_method(df_ok, primary_metric=PRIMARY_METRIC)
comparison_table = build_comparison_table(best)
comparison_table

,method,QPS,avg_latency_ms,n_queries,m,ef_construct,ef_search,rrf_k,hnsw_collection,avg_latency_s,MRR@10,NDCG@10,Precision@1,Precision@10,Precision@3,Precision@5,Recall@10
0,hnsw,83.64,11.9554,500,64.0,300.0,100.0,NaN,NaN,0.011955,0.508400,0.772311,0.018,0.8872,0.668667,0.7932,0.8872
1,hybrid_rrf,31.94,31.3114,500,NaN,NaN,NaN,240.0,thesis_bench_hnsw_default,0.031311,0.475167,0.494265,0.012,0.5410,0.538667,0.5772,0.5410


In [19]:
# Trade-off between speed (QPS) and quality (Precision@10): Pareto frontier

if all(c in df_ok.columns for c in ("QPS", PRIMARY_METRIC)):
    pareto = pareto_frontier(df_ok, x="QPS", y=PRIMARY_METRIC, maximize_x=True, maximize_y=True)
    pareto
else:
    pareto = pd.DataFrame()
    pareto

In [20]:
# Export CSV/LaTeX + Markdown report

outputs = export_outputs(df_all, out_dir=OUTPUT_DIR, primary_metric=PRIMARY_METRIC)
outputs

{'comparison_csv': WindowsPath('D:/P_work/Rag-VKR/experiments/results/comparison_table.csv'),
 'comparison_latex': WindowsPath('D:/P_work/Rag-VKR/experiments/results/comparison_table.tex'),
 'report_md': WindowsPath('D:/P_work/Rag-VKR/experiments/results/report.md'),
 'fig_dir': WindowsPath('D:/P_work/Rag-VKR/experiments/results/figures')}

In [21]:
# Load exported comparison table (sanity check)

pd.read_csv(outputs["comparison_csv"]).head()

,method,QPS,avg_latency_ms,n_queries,m,ef_construct,ef_search,rrf_k,hnsw_collection,avg_latency_s,MRR@10,NDCG@10,Precision@1,Precision@10,Precision@3,Precision@5,Recall@10
0,hnsw,83.64,11.9554,500,64.0,300.0,100.0,NaN,NaN,0.011955,0.508400,0.772311,0.018,0.8872,0.668667,0.7932,0.8872
1,hybrid_rrf,31.94,31.3114,500,NaN,NaN,NaN,240.0,thesis_bench_hnsw_default,0.031311,0.475167,0.494265,0.012,0.5410,0.538667,0.5772,0.5410
